In [14]:
from __future__ import annotations

import os
import operator
from typing import TypedDict,List,Annotated

from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage

from dotenv import load_dotenv


In [15]:
load_dotenv()

True

In [17]:
class Task(BaseModel):
    id:int
    title:str
    brief:str=Field(...,description="What to cover")

In [16]:
class Plan(BaseModel):
    blog_title:str
    tasks:List[Task]

In [18]:
class State(TypedDict):
    topic:str
    plan:Plan
    section:Annotated[List[str],operator.add]
    final:str

In [19]:
llm=ChatOpenAI(model='gpt-5.4-mini')

In [20]:
def orchestrator(state:State)->dict:
    plan=llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic:{state['topic']}")
        ]
    )
    return {'plan':plan}

In [ ]:
def fanout(state:State):
    return [Send('worker',{'task':task,'topic':state['topic'],'plan':state['plan']})
            for task in state['plan'].tasks]